# GraphFlow (Workflows)

In this section you'll learn how to create an _multi-agent workflow_ using {py:class}`~autogen_agentchat.teams.GraphFlow`, or simply "flow" for short.
It uses structured execution and precisely controls how agents interact to accomplish a task.

We'll first show you how to create and run a flow. We'll then explain how to observe and debug flow behavior, 
and discuss important operations for managing execution.

AutoGen AgentChat provides a team for directed graph execution:

- {py:class}`~autogen_agentchat.teams.GraphFlow`: A team that follows a {py:class}`~autogen_agentchat.teams.DiGraph`
to control the execution flow between agents. 
Supports sequential, parallel, conditional, and looping behaviors.

```{note}
**When should you use {py:class}`~autogen_agentchat.teams.GraphFlow`?**

Use Graph when you need strict control over the order in which agents act, or when different outcomes must lead to different next steps.
Start with a simple team such as {py:class}`~autogen_agentchat.teams.RoundRobinGroupChat` or {py:class}`~autogen_agentchat.teams.SelectorGroupChat`
if ad-hoc conversation flow is sufficient. 
Transition to a structured workflow when your task requires deterministic control,
conditional branching, or handling complex multi-step processes with cycles.
```

> **Warning:** {py:class}`~autogen_agentchat.teams.GraphFlow` is an **experimental feature**. 
Its API, behavior, and capabilities are **subject to change** in future releases.

## Creating and Running a Flow

{py:class}`~autogen_agentchat.teams.DiGraphBuilder` is a fluent utility that lets you easily construct execution graphs for workflows. It supports building:

- Sequential chains
- Parallel fan-outs
- Conditional branching
- Loops with safe exit conditions

Each node in the graph represents an agent, and edges define the allowed execution paths. Edges can optionally have conditions based on agent messages.

### Sequential Flow

We will begin by creating a simple workflow where a **writer** drafts a paragraph and a **reviewer** provides feedback. This graph terminates after the reviewer comments on the writer. 

Note, the flow automatically computes all the source and leaf nodes of the graph and the execution starts at all the source nodes in the graph and completes execution when no nodes are left to execute.

In [24]:
import os
from dotenv import load_dotenv

load_dotenv()

print("OPENAI_API_KEY:", os.getenv("OPENAI_API_KEY"))
print("AZURE_OPENAI_API_KEY:", os.getenv("AZURE_OPENAI_API_KEY"))
print("AZURE_OPENAI_ENDPOINT:", os.getenv("AZURE_OPENAI_ENDPOINT"))
print("AZURE_OPENAI_API_VERSION:", os.getenv("AZURE_OPENAI_API_VERSION"))

OPENAI_API_KEY: 
AZURE_OPENAI_API_KEY: 4LHtl13OVMOXTRnlrlzb2WA7D5WkG7twEl9BbTv1KloAsOugm5u0JQQJ99BBACfhMk5XJ3w3AAAAACOGZwSR
AZURE_OPENAI_ENDPOINT: https://ai-aiagenthub470987418747.openai.azure.com/
AZURE_OPENAI_API_VERSION: 2025-01-01-preview


In [27]:
import os
from dotenv import load_dotenv
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient

load_dotenv()

# Create an OpenAI model client


client = AzureOpenAIChatCompletionClient(
    model="gpt-4.1-nano",
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),  # or "2025-01-01-preview"
    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)
# Create the writer agent
writer = AssistantAgent("writer", model_client=client, system_message="Draft a short paragraph on climate change.")

# Create the reviewer agent
reviewer = AssistantAgent("reviewer", model_client=client, system_message="Review the draft and suggest improvements.")

# Build the graph
builder = DiGraphBuilder()
builder.add_node(writer).add_node(reviewer)
builder.add_edge(writer, reviewer)

# Build and validate the graph
graph = builder.build()
builder.set_entry_point(writer)
# Create the flow
flow = GraphFlow([writer, reviewer], graph=graph)

In [28]:
# Use `asyncio.run(...)` and wrap the below in a async function when running in a script.
stream = flow.run_stream(task="Write a short paragraph about climate change.")
async for event in stream:  # type: ignore
    print(event)
# Use Console(flow.run_stream(...)) for better formatting in console.

source='user' models_usage=None metadata={} content='Write a short paragraph about climate change.' type='TextMessage'
source='writer' models_usage=RequestUsage(prompt_tokens=28, completion_tokens=95) metadata={} content='Climate change refers to long-term shifts in temperature, weather patterns, and global ecosystems primarily caused by human activities such as burning fossil fuels, deforestation, and industrial processes. These actions increase greenhouse gas concentrations in the atmosphere, leading to global warming and adverse effects like rising sea levels, extreme weather events, and loss of biodiversity. Addressing climate change requires coordinated efforts to reduce emissions, transition to renewable energy sources, and implement sustainable practices to protect the planet for future generations.' type='TextMessage'
source='reviewer' models_usage=RequestUsage(prompt_tokens=127, completion_tokens=161) metadata={} content='This is a solid overview of climate change. To enhance 

### Parallel Flow with Join

We now create a slightly more complex flow:

- A **writer** drafts a paragraph.
- Two **editors** independently edit for grammar and style (parallel fan-out).
- A **final reviewer** consolidates their edits (join).

Execution starts at the **writer**, fans out to **editor1** and **editor2** simultaneously, and then both feed into the **final reviewer**.


In [31]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient

# Create an OpenAI model client
client = AzureOpenAIChatCompletionClient(
    model="gpt-4.1-nano",
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),  # or "2025-01-01-preview"
    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)
# Create the writer agent
writer = AssistantAgent("writer", model_client=client, system_message="Draft a short paragraph on climate change.")

# Create two editor agents
editor1 = AssistantAgent("editor1", model_client=client, system_message="Edit the paragraph for grammar.")

editor2 = AssistantAgent("editor2", model_client=client, system_message="Edit the paragraph for style.")

# Create the final reviewer agent
final_reviewer = AssistantAgent(
    "final_reviewer",
    model_client=client,
    system_message="Consolidate the grammar and style edits into a final version.",
)

# Build the workflow graph
builder = DiGraphBuilder()
builder.add_node(writer).add_node(editor1).add_node(editor2).add_node(final_reviewer)

# Fan-out from writer to editor1 and editor2
builder.add_edge(writer, editor1)
builder.add_edge(writer, editor2)

# Fan-in both editors into final reviewer
builder.add_edge(editor1, final_reviewer)
builder.add_edge(editor2, final_reviewer)

# Build and validate the graph
graph = builder.build()

# Create the flow
flow = GraphFlow(
    participants=builder.get_participants(),
    graph=graph,
)

# Run the workflow
await Console(flow.run_stream(task="Write a short paragraph about climate change."))

---------- TextMessage (user) ----------
Write a short paragraph about climate change.
---------- TextMessage (writer) ----------
Climate change refers to significant and lasting changes in global weather patterns, primarily caused by the increase in greenhouse gases such as carbon dioxide and methane due to human activities like burning fossil fuels, deforestation, and industrial processes. These changes lead to rising temperatures, melting glaciers, more frequent and severe storms, and shifts in ecosystems, posing serious threats to biodiversity, agriculture, and human health. Addressing climate change requires global cooperation to reduce emissions, transition to renewable energy sources, and implement sustainable practices to protect the planet for future generations.
---------- TextMessage (editor1) ----------
Climate change refers to significant and lasting changes in global weather patterns, primarily caused by the increase in greenhouse gases such as carbon dioxide and methane 

TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, content='Write a short paragraph about climate change.', type='TextMessage'), TextMessage(source='writer', models_usage=RequestUsage(prompt_tokens=28, completion_tokens=106), metadata={}, content='Climate change refers to significant and lasting changes in global weather patterns, primarily caused by the increase in greenhouse gases such as carbon dioxide and methane due to human activities like burning fossil fuels, deforestation, and industrial processes. These changes lead to rising temperatures, melting glaciers, more frequent and severe storms, and shifts in ecosystems, posing serious threats to biodiversity, agriculture, and human health. Addressing climate change requires global cooperation to reduce emissions, transition to renewable energy sources, and implement sustainable practices to protect the planet for future generations.', type='TextMessage'), TextMessage(source='editor1', models_usage=Requ

## Message Filtering

### Execution Graph vs. Message Graph

In {py:class}`~autogen_agentchat.teams.GraphFlow`, the **execution graph** is defined using 
{py:class}`~autogen_agentchat.teams.DiGraph`, which controls the order in which agents execute.
However, the execution graph does not control what messages an agent receives from other agents.
By default, all messages are sent to all agents in the graph.

**Message filtering** is a separate feature that allows you to filter the messages
received by each agent and limiting their model context to only the relevant information.
The set of message filters defines the **message graph** in the flow.

Specifying the message graph can help with:
- Reduce hallucinations
- Control memory load
- Focus agents only on relevant information

You can use {py:class}`~autogen_agentchat.agents.MessageFilterAgent` together with {py:class}`~autogen_agentchat.agents.MessageFilterConfig` and {py:class}`~autogen_agentchat.agents.PerSourceFilter` to define these rules.

In [36]:
from autogen_agentchat.agents import AssistantAgent, MessageFilterAgent, MessageFilterConfig, PerSourceFilter
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient

# Model client
client = AzureOpenAIChatCompletionClient(
    model="gpt-4.1-nano",
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),  # or "2025-01-01-preview"
    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)
# Create agents
researcher = AssistantAgent(
    "researcher", model_client=client, system_message="Summarize key facts about climate change."
)
analyst = AssistantAgent("analyst", model_client=client, system_message="Review the summary and suggest improvements.")
presenter = AssistantAgent(
    "presenter", model_client=client, system_message="Prepare a presentation slide based on the final summary."
)

# Apply message filtering
filtered_analyst = MessageFilterAgent(
    name="analyst",
    wrapped_agent=analyst,
    filter=MessageFilterConfig(per_source=[PerSourceFilter(source="researcher", position="last", count=1)]),
)

filtered_presenter = MessageFilterAgent(
    name="presenter",
    wrapped_agent=presenter,
    filter=MessageFilterConfig(per_source=[PerSourceFilter(source="analyst", position="last", count=1)]),
)

# Build the flow
builder = DiGraphBuilder()
builder.add_node(researcher).add_node(filtered_analyst).add_node(filtered_presenter)
builder.add_edge(researcher, filtered_analyst).add_edge(filtered_analyst, filtered_presenter)

# Create the flow
flow = GraphFlow(
    participants=builder.get_participants(),
    graph=builder.build(),
)

# Run the flow
await Console(flow.run_stream(task="Summarize key facts about climate change."))

---------- TextMessage (user) ----------
Summarize key facts about climate change.
---------- TextMessage (researcher) ----------
Climate change refers to long-term shifts in temperature, precipitation, and other atmospheric conditions due to human activities, primarily the burning of fossil fuels. Key facts include:

1. Global temperatures are rising, with recent decades being the warmest on record.
2. Greenhouse gases such as carbon dioxide (CO₂) and methane (CH₄) trap heat in the atmosphere, leading to the "greenhouse effect."
3. Human activities, including deforestation and industrial processes, have significantly increased greenhouse gas concentrations since the Industrial Revolution.
4. Climate change is causing more frequent and severe weather events, such as hurricanes, droughts, and heavy rainfall.
5. Rising temperatures contribute to melting glaciers and ice caps, leading to sea level rise.
6. Changes in climate threaten biodiversity, agriculture, water resources, and human h

TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, content='Summarize key facts about climate change.', type='TextMessage'), TextMessage(source='researcher', models_usage=RequestUsage(prompt_tokens=30, completion_tokens=221), metadata={}, content='Climate change refers to long-term shifts in temperature, precipitation, and other atmospheric conditions due to human activities, primarily the burning of fossil fuels. Key facts include:\n\n1. Global temperatures are rising, with recent decades being the warmest on record.\n2. Greenhouse gases such as carbon dioxide (CO₂) and methane (CH₄) trap heat in the atmosphere, leading to the "greenhouse effect."\n3. Human activities, including deforestation and industrial processes, have significantly increased greenhouse gas concentrations since the Industrial Revolution.\n4. Climate change is causing more frequent and severe weather events, such as hurricanes, droughts, and heavy rainfall.\n5. Rising temperatures contr

## 🔁 Advanced Example: Conditional Loop + Filtered Summary

This example demonstrates:

- A loop between generator and reviewer (which exits when reviewer says "APPROVE")
- A summarizer agent that only sees the first user input and the last reviewer message


In [40]:
from autogen_agentchat.agents import AssistantAgent, MessageFilterAgent, MessageFilterConfig, PerSourceFilter
from autogen_agentchat.teams import (
    DiGraphBuilder,
    GraphFlow,
)
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient

# Model client
model_client = AzureOpenAIChatCompletionClient(
    model="gpt-4.1-nano",
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),  # or "2025-01-01-preview"
    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)
# Agents
generator = AssistantAgent("generator", model_client=model_client, system_message="Generate a list of creative ideas.")
reviewer = AssistantAgent(
    "reviewer",
    model_client=model_client,
    system_message="Review ideas and say 'REVISE' and provide feedbacks, or 'APPROVE' for final approval.",
)
summarizer_core = AssistantAgent(
    "summary", model_client=model_client, system_message="Summarize the user request and the final feedback."
)

# Filtered summarizer
filtered_summarizer = MessageFilterAgent(
    name="summary",
    wrapped_agent=summarizer_core,
    filter=MessageFilterConfig(
        per_source=[
            PerSourceFilter(source="user", position="first", count=1),
            PerSourceFilter(source="reviewer", position="last", count=1),
        ]
    ),
)

# Build graph with conditional loop
builder = DiGraphBuilder()
builder.add_node(generator).add_node(reviewer).add_node(filtered_summarizer)
builder.add_edge(generator, reviewer)
builder.add_edge(reviewer, generator, condition="REVISE")
builder.add_edge(reviewer, filtered_summarizer, condition="APPROVE")
builder.set_entry_point(generator)  # Set entry point to generator. Required if there are no source nodes.
graph = builder.build()

# Create the flow
flow = GraphFlow(
    participants=builder.get_participants(),
    graph=graph,
)

# Run the flow and pretty print the output in the console
await Console(flow.run_stream(task="Brainstorm ways to reduce plastic waste."))

---------- TextMessage (user) ----------
Brainstorm ways to reduce plastic waste.
---------- TextMessage (generator) ----------
Certainly! Here are some creative and practical ideas to reduce plastic waste:

1. **Biodegradable Packaging:** Encourage businesses to switch to biodegradable or compostable packaging materials instead of traditional plastics.

2. **Reusable Shopping Bags:** Promote the use of sturdy, stylish reusable bags made from cloth, jute, or recycled materials.

3. **Refill Stations:** Install refill stations for products like shampoo, soap, and cleaning supplies to eliminate single-use plastic bottles.

4. **Plastic-Free Products:** Develop and market products that are entirely plastic-free, such as bar shampoos, soaps, and solid cleansers.

5. **Community Swap Events:** Organize local events where people can exchange or donate used plastic items that can be repurposed or recycled.

6. **Innovative Recycling Technologies:** Invest in and support new technologies that 

TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, content='Brainstorm ways to reduce plastic waste.', type='TextMessage'), TextMessage(source='generator', models_usage=RequestUsage(prompt_tokens=27, completion_tokens=545), metadata={}, content='Certainly! Here are some creative and practical ideas to reduce plastic waste:\n\n1. **Biodegradable Packaging:** Encourage businesses to switch to biodegradable or compostable packaging materials instead of traditional plastics.\n\n2. **Reusable Shopping Bags:** Promote the use of sturdy, stylish reusable bags made from cloth, jute, or recycled materials.\n\n3. **Refill Stations:** Install refill stations for products like shampoo, soap, and cleaning supplies to eliminate single-use plastic bottles.\n\n4. **Plastic-Free Products:** Develop and market products that are entirely plastic-free, such as bar shampoos, soaps, and solid cleansers.\n\n5. **Community Swap Events:** Organize local events where people can exch